# Silver Layer - Data Cleaning & Transformation

## Purpose 
Reads from the Bronze Delta table, applies cleaning and 
transformation logic, and writes to the Silver Delta table.

## Reads from
- `pipeline_bronze` — raw ingested data

## Writes to
- 'pipeline_silver' - cleaned and transformed sales data

## Notes
- Standardizes city and category casing
- Replaces null categories with "Unknown"
- Casts order_date to DateType
- Adds total_value and order_size columns
- Filters out orders under $25


In [0]:
dbutils.widgets.text("run_date", "2024-01-01", "Run Date")

run_date = dbutils.widgets.get("run_date")

SOURCE_TABLE = "pipeline_bronze"
TARGET_TABLE = "pipeline_silver"

print(f"Reading from: {SOURCE_TABLE}")
print(f"Writing to:   {TARGET_TABLE}")
print(f"Run date:     {run_date}")

In [0]:
from pyspark.sql.functions import col, round, when, lit, initcap, current_timestamp
from pyspark.sql.types import DateType

In [0]:
df_bronze = spark.read.table(SOURCE_TABLE)
print(f"Rows read from Bronze: {df_bronze.count()}")
df_bronze.show(5)

In [0]:
df_cleaned = (
    df_bronze
    .withColumn("city", initcap(col("city")))
    .withColumn("category", initcap(col("category")))
    .fillna({"category" : "Unknown"})
    .withColumn("order_date", col("order_date").cast(DateType()))
    .withColumn("total_value", round(col("amount") * col("quantity"), 2))
    .withColumn("order_size", when(col("amount") >= 500, "large").otherwise("small"))
    .filter(col("amount") >= 25)
    .drop("env")
)

df_cleaned.show()

In [0]:
#validate
before = df_bronze.count()
after = df_cleaned.count()
print(f"Rows before filtering: {before}")
print(f"Rows after filtering:  {after}")
print(f"Rows removed:          {before - after}")
if after == 0:
    raise Exception("SILVER FAILED: No rows after cleaning - aborting")

print("Validation passed - writing to Silver")

In [0]:
df_cleaned.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("pipeline_silver")

final_count = spark.read.table("pipeline_silver").count()
print(f"pipeline_silver written — {final_count} rows")